# 02 - Exploração de Dados via UtilCkan

Notebook **independente** do experimento principal — não depende de nenhum parquet pré-existente.

Obtém diretamente do Portal de Dados Abertos do STJ:
1. **Espelhos de acórdãos** (tese jurídica, tema, referências, etc.) dos JSONs por órgão julgador.
2. **Inteiro Teor** (opcional) de cada acórdão, dos ZIPs de texto integral.

A `UtilCkan` aceita os seguintes filtros no construtor:
- `anos` — anos de publicação (ex: `{'2023', '2024'}`)
- `datas` — datas de publicação (ex: `{'2024-08-22', '20240822'}`)
- `classes` — siglas de classes processuais (ex: `{'HC', 'RHC'}`)
- `orgaos` — siglas dos órgãos julgadores (ex: `['T5', 'T6', 'S3']`)
- `registros` — números de registro; aceita `str`, tupla `(registro, data_pub)` ou `(registro, data_pub, tipo_decisao)`
- `documentos` — `seq_documento_acordao` (íntegras específicas)

Saída: `espelhos_acordaos_por_ano_com_texto.parquet`


## 1. Configuração e Imports

In [31]:
import os, sys
from pathlib import Path
import pandas as pd
from util_ckan import UtilCkan, UtilCkanIntegra

# ─── Diretórios de cache ──────────────────────────────────────────────────────
# As subpastas espelhos/, integras/ e metadados_integras/ são criadas automaticamente pela UtilCkan
DOWNLOAD_DIR  = Path('downloads_stj')
PASTA_DATA    = '../data'
PARQUET_SAIDA = os.path.join(PASTA_DATA, 'espelhos_acordaos_por_ano_com_texto.parquet')

CKAN_TIMEOUT           = 600
ATUALIZAR_CACHE_E_MAPAS = 12 * 60 # 12h entre atualizações ou se os mapas não existirem

print(f'Download dir          : {DOWNLOAD_DIR.resolve()}')
print(f'CKAN_TIMEOUT          : {CKAN_TIMEOUT}')
print(f'ATUALIZAR_CACHE_E_MAPAS: {ATUALIZAR_CACHE_E_MAPAS}')


Download dir          : /mnt/d/wsl_pucdev/agent-orchestration-2026/notebooks/downloads_stj
CKAN_TIMEOUT          : 600
ATUALIZAR_CACHE_E_MAPAS: 720


## 2. Exemplos de Filtros da UtilCkan

Cada célula abaixo demonstra um cenário independente de filtro.  
**Execute apenas a célula do cenário desejado** — todas usam `atualizar_cache_e_mapas=False`  
para operar apenas sobre o cache local (sem novos downloads).


### 2.1 Filtro por Ano de Publicação

In [32]:
# ── Filtro: apenas acórdãos publicados nos anos especificados ─────────────────
# Aplica-se sobre todos os órgãos julgadores disponíveis no cache local.

ckan = UtilCkan(
    anos    = {'2023', '2024'},          # ← ajuste os anos desejados
    download_dir  = DOWNLOAD_DIR,        # ← opcional - padrão é 'downloads_stj'
    timeout       = CKAN_TIMEOUT,        # ← opcional - padrão é 600
    atualizar_cache_e_mapas = ATUALIZAR_CACHE_E_MAPAS,   # ← opcional - padrão é 12h
)

print(f'Total no mapa de espelhos: {len(ckan._mapa_espelhos)}')
registros = ckan.consultar_mapa()
print(f'Registros após filtro por ano (2023/2024): {len(registros)}')

df = ckan.gerar_dataset_espelhos(
    caminho_saida    = PARQUET_SAIDA.replace('.parquet', '_ex_anos.parquet'),
    incluir_integras = False,
    incluir_ementas  = True,
    incluir_decisoes = False,
)
UtilCkan.exibir_amostra(df, n=2, titulo='Acórdãos filtrados por ano 2023/2024')


  📋  Mapa espelhos carregado: 120239 registros
  📋  Mapa íntegras carregado: 2465080 registros
Total no mapa de espelhos: 120239
Registros após filtro por ano (2023/2024): 61210
Registros cruzados (espelho × íntegra): 61210


Espelhos:   0%|          | 0/259 [00:00<?, ?it/s]


Registros extraídos: 61784
───────────────────────────────────────────────────────
  📄  ../data/espelhos_acordaos_por_ano_com_texto_ex_anos.parquet
      62.52 MB | 23 colunas | 61210 registros
───────────────────────────────────────────────────────
  📅  Por ano de publicação:
      2023 → 28775 registros | integra: 0
      2024 → 32435 registros | integra: 0
───────────────────────────────────────────────────────
  ⚖️  Top classes:
      AgRg no HC                          13466
      AgInt no AREsp                      11812
      AgInt no REsp                        7105
      AgRg no AREsp                        4797
      AgRg no RHC                          2478
      REsp                                 2199
      HC                                   1997
      AgRg no REsp                         1896
      AgInt nos EDcl no AREsp              1340
      AgInt nos EDcl no REsp               1004
───────────────────────────────────────────────────────
  ✅  Concluído!

Acórdãos 

### 2.2 Filtro por Órgão Julgador

In [33]:
# ── Filtro: órgãos específicos (siglas conforme CKAN do STJ) ─────────────────
# Órgãos disponíveis: CE, S1, S2, S3, T1, T2, T3, T4, T5, T6

ckan = UtilCkan(
    orgaos  = ['T5', 'T6'],              # ← Quinta e Sexta Turmas
    anos    = {'2023'},                  # ← ajuste os anos desejados
    download_dir  = DOWNLOAD_DIR,        # ← opcional - padrão é 'downloads_stj'
    timeout       = CKAN_TIMEOUT,        # ← opcional - padrão é 600
    atualizar_cache_e_mapas = ATUALIZAR_CACHE_E_MAPAS,   # ← opcional - padrão é 12h
)

registros = ckan.consultar_mapa()
print(f'Registros após filtro T5+T6 (2023): {len(registros)}')

# Cruzamento com mapa de íntegras (sem download)
cruzados = ckan.cruzar_espelhos_integras()
tem_integra = [r for r in cruzados if r.get('tem_integra')]
print(f'Desses, com íntegra indexada: {len(tem_integra)}')

df = ckan.gerar_dataset_espelhos(
    caminho_saida    = PARQUET_SAIDA.replace('.parquet', '_ex_orgaos.parquet'),
    incluir_integras = False,
    incluir_ementas  = True,
    incluir_decisoes = False,
)
UtilCkan.exibir_amostra(df, n=2, titulo='Acórdãos T5 e T6 (2023)')


  📋  Mapa espelhos carregado: 120239 registros
  📋  Mapa íntegras carregado: 2465080 registros
Registros após filtro T5+T6 (2023): 11950
Desses, com íntegra indexada: 8459
Registros cruzados (espelho × íntegra): 11950


Espelhos:   0%|          | 0/29 [00:00<?, ?it/s]


Registros extraídos: 12041
───────────────────────────────────────────────────────
  📄  ../data/espelhos_acordaos_por_ano_com_texto_ex_orgaos.parquet
      13.38 MB | 23 colunas | 11950 registros
───────────────────────────────────────────────────────
  📅  Por ano de publicação:
      2023 → 11950 registros | integra: 0
───────────────────────────────────────────────────────
  ⚖️  Top classes:
      AgRg no HC                           6047
      AgRg no AREsp                        2190
      AgRg no RHC                          1085
      AgRg no REsp                          979
      HC                                    224
      AgRg nos EDcl no HC                   128
      AgRg no AgRg no AREsp                 120
      EDcl no AgRg no AREsp                 106
      AgRg nos EDcl no AREsp                101
      RHC                                    87
───────────────────────────────────────────────────────
  ✅  Concluído!

Acórdãos T5 e T6 (2023) (11950 registros totais, 

### 2.3 Filtro por Classe Processual

In [34]:
# ── Filtro: classes processuais específicas ───────────────────────────────────
# Exemplos de classes: 'HC' (Habeas Corpus), 'RHC', 'REsp', 'AgRg', 'AREsp', etc.

ckan = UtilCkan(
    classes = {'HC', 'RHC'},             # ← ajuste as classes desejadas
    anos    = {'2023'},                  # ← ajuste os anos desejados
    download_dir  = DOWNLOAD_DIR,        # ← opcional - padrão é 'downloads_stj'
    timeout       = CKAN_TIMEOUT,        # ← opcional - padrão é 600
    atualizar_cache_e_mapas = ATUALIZAR_CACHE_E_MAPAS,   # ← opcional - padrão é 12h
)

registros = ckan.consultar_mapa()
print(f'Registros HC/RHC (2023): {len(registros)}')

df = ckan.gerar_dataset_espelhos(
    caminho_saida    = PARQUET_SAIDA.replace('.parquet', '_ex_classes.parquet'),
    incluir_integras = False,
    incluir_ementas  = True,
    incluir_decisoes = False,
)
UtilCkan.exibir_amostra(df, n=2, titulo='Acórdãos HC/RHC (2023)')


  📋  Mapa espelhos carregado: 120239 registros
  📋  Mapa íntegras carregado: 2465080 registros
Registros HC/RHC (2023): 8098
Registros cruzados (espelho × íntegra): 8098


Espelhos:   0%|          | 0/69 [00:00<?, ?it/s]


Registros extraídos: 8125
───────────────────────────────────────────────────────
  📄  ../data/espelhos_acordaos_por_ano_com_texto_ex_classes.parquet
      8.95 MB | 23 colunas | 8098 registros
───────────────────────────────────────────────────────
  📅  Por ano de publicação:
      2023 →  8098 registros | integra: 0
───────────────────────────────────────────────────────
  ⚖️  Top classes:
      AgRg no HC                           6049
      AgRg no RHC                          1085
      HC                                    257
      AgRg nos EDcl no HC                   128
      RHC                                    98
      AgRg no AgRg no HC                     82
      EDcl no AgRg no HC                     78
      AgRg nos EDcl no RHC                   53
      RCD no HC                              46
      EDcl no HC                             34
───────────────────────────────────────────────────────
  ✅  Concluído!

Acórdãos HC/RHC (2023) (8098 registros totais, exib

### 2.4 Filtro por Data de Publicação

In [35]:
# ── Filtro: datas de publicação específicas ────────────────────────────────────
# Aceita vários formatos: 'YYYY-MM-DD', 'DD/MM/YYYY', 'YYYYMMDD'.
# Útil para montar datasets de dias específicos de publicação no DJE.

ckan = UtilCkan(
    datas   = {'2023-06-01', '01/06/2023', '20230615'},  # ← ajuste as datas desejadas
    download_dir  = DOWNLOAD_DIR,
    timeout       = CKAN_TIMEOUT,
    atualizar_cache_e_mapas = ATUALIZAR_CACHE_E_MAPAS,
)

registros = ckan.consultar_mapa()
print(f'Registros nas datas informadas: {len(registros)}')

df = ckan.gerar_dataset_espelhos(
    caminho_saida    = PARQUET_SAIDA.replace('.parquet', '_ex_datas.parquet'),
    incluir_integras = False,
    incluir_ementas  = True,
    incluir_decisoes = False,
)
UtilCkan.exibir_amostra(df, n=2, titulo='Acórdãos por datas de publicação')

  📋  Mapa espelhos carregado: 120239 registros
  📋  Mapa íntegras carregado: 2465080 registros
Registros nas datas informadas: 451
Registros cruzados (espelho × íntegra): 451


Espelhos:   0%|          | 0/13 [00:00<?, ?it/s]


Registros extraídos: 455
───────────────────────────────────────────────────────
  📄  ../data/espelhos_acordaos_por_ano_com_texto_ex_datas.parquet
      0.51 MB | 23 colunas | 451 registros
───────────────────────────────────────────────────────
  📅  Por ano de publicação:
      2023 →   451 registros | integra: 0
───────────────────────────────────────────────────────
  ⚖️  Top classes:
      AgInt no AREsp                        136
      AgRg no HC                             88
      AgInt no REsp                          75
      AgRg no RHC                            22
      AgInt nos EDcl no AREsp                13
      AgInt nos EDcl no REsp                  8
      EDcl no AgInt no REsp                   8
      REsp                                    8
      EDcl no AgInt no AREsp                  7
      AgInt nos EAREsp                        6
───────────────────────────────────────────────────────
  ✅  Concluído!

Acórdãos por datas de publicação (451 registros totais,

### 2.5 Filtro por Número de Registro

In [36]:
# ── Filtro: números de registro específicos (sem data/tipo) ───────────────────
# Útil quando se conhece apenas o numeroRegistro, sem a data de publicação.
# A UtilCkan retornará TODOS os acórdãos desses registros (qualquer data/tipo).

registros_alvo = {
    '202202853462',    # ← substitua pelos registros de interesse
    '202201555326',
    '202201341093',
}

ckan = UtilCkan(
    registros     = registros_alvo,      # ← ajuste os registros de interesse
    download_dir  = DOWNLOAD_DIR,        # ← opcional - padrão é 'downloads_stj'
    timeout       = CKAN_TIMEOUT,        # ← opcional - padrão é 600
    atualizar_cache_e_mapas = ATUALIZAR_CACHE_E_MAPAS,   # ← opcional - padrão é 12h
)

encontrados = ckan.consultar_mapa()
print(f'Registros encontrados no mapa: {len(encontrados)}')
for r in encontrados:
    print(f"  {r['numero_registro']}  |  pub: {r['data_publicacao']}  |  tipo: {r['tipo_decisao']}  |  órgão: {r['orgao']}")

df = ckan.gerar_dataset_espelhos(
    caminho_saida    = PARQUET_SAIDA.replace('.parquet', '_ex_registros.parquet'),
    incluir_integras = False,
    incluir_ementas  = True,
    incluir_decisoes = True,
)
UtilCkan.exibir_amostra(df, n=3, titulo='Acórdãos por número de registro')


  📋  Mapa espelhos carregado: 120239 registros
  📋  Mapa íntegras carregado: 2465080 registros
Registros encontrados no mapa: 3
  202202853462  |  pub: 20230601  |  tipo: ACÓRDÃO  |  órgão: S3
  202201555326  |  pub: 20220620  |  tipo: ACÓRDÃO  |  órgão: T5
  202201341093  |  pub: 20220615  |  tipo: ACÓRDÃO  |  órgão: T5
Registros cruzados (espelho × íntegra): 3


Espelhos:   0%|          | 0/2 [00:00<?, ?it/s]


Registros extraídos: 3
───────────────────────────────────────────────────────
  📄  ../data/espelhos_acordaos_por_ano_com_texto_ex_registros.parquet
      0.04 MB | 24 colunas | 3 registros
───────────────────────────────────────────────────────
  📅  Por ano de publicação:
      2022 →     2 registros | integra: 0
      2023 →     1 registros | integra: 0
───────────────────────────────────────────────────────
  ⚖️  Top classes:
      HC                                      2
      AgRg no HC                              1
───────────────────────────────────────────────────────
  ✅  Concluído!

Acórdãos por número de registro (3 registros totais, exibindo 3):
═════════════════════════════════════════════════════════════════
  dataDecisao                 : 20230510
  dataPublicacao              : DJE        DATA:01/06/2023
  data_publicacao_iso         : 20230601
  descricaoClasse             : HABEAS CORPUS
  id                          : 000844657
  id_mapa                     : 2022

### 2.6 Filtro por Registro com Data de Publicação

In [37]:
# ── Filtro: tuplas (registro, data_publicacao) ────────────────────────────────
# Permite identificar acórdãos de forma precisa quando se conhece o número
# de registro E a data de publicação no DJE.
# A data aceita vários formatos: YYYYMMDD, YYYY-MM-DD, DD/MM/YYYY.

registros_com_data = {
    ('202202853462', '2023-06-01'),   # formato ISO
    ('202201555326', '20220620'),     # formato YYYYMMDD
    ('202201341093', '15/06/2022'),   # formato DD/MM/YYYY
}

ckan = UtilCkan(
    registros     = registros_com_data,  # ← tuplas (registro, data_publicacao)
    download_dir  = DOWNLOAD_DIR,        # ← opcional - padrão é 'downloads_stj'
    timeout       = CKAN_TIMEOUT,        # ← opcional - padrão é 600
    atualizar_cache_e_mapas = ATUALIZAR_CACHE_E_MAPAS,   # ← opcional - padrão é 12h
)

encontrados = ckan.consultar_mapa()
print(f'Registros encontrados (registro + data): {len(encontrados)}')
for r in encontrados:
    print(f"  {r['numero_registro']}  |  pub: {r['data_publicacao']}  |  tipo: {r['tipo_decisao']}")

df = ckan.gerar_dataset_espelhos(
    caminho_saida    = PARQUET_SAIDA.replace('.parquet', '_ex_reg_data.parquet'),
    incluir_integras = False,
    incluir_ementas  = True,
    incluir_decisoes = True,
)
UtilCkan.exibir_amostra(df, n=3, titulo='Acórdãos por registro + data de publicação')


  📋  Mapa espelhos carregado: 120239 registros
  📋  Mapa íntegras carregado: 2465080 registros
Registros encontrados (registro + data): 3
  202202853462  |  pub: 20230601  |  tipo: ACÓRDÃO
  202201555326  |  pub: 20220620  |  tipo: ACÓRDÃO
  202201341093  |  pub: 20220615  |  tipo: ACÓRDÃO
Registros cruzados (espelho × íntegra): 3


Espelhos:   0%|          | 0/2 [00:00<?, ?it/s]


Registros extraídos: 3
───────────────────────────────────────────────────────
  📄  ../data/espelhos_acordaos_por_ano_com_texto_ex_reg_data.parquet
      0.04 MB | 24 colunas | 3 registros
───────────────────────────────────────────────────────
  📅  Por ano de publicação:
      2022 →     2 registros | integra: 0
      2023 →     1 registros | integra: 0
───────────────────────────────────────────────────────
  ⚖️  Top classes:
      HC                                      2
      AgRg no HC                              1
───────────────────────────────────────────────────────
  ✅  Concluído!

Acórdãos por registro + data de publicação (3 registros totais, exibindo 3):
═════════════════════════════════════════════════════════════════
  dataDecisao                 : 20230510
  dataPublicacao              : DJE        DATA:01/06/2023
  data_publicacao_iso         : 20230601
  descricaoClasse             : HABEAS CORPUS
  id                          : 000844657
  id_mapa                 

### 2.7 Filtro por Registro, Data e Tipo de Decisão

In [38]:
# ── Filtro: tuplas (registro, data_publicacao, tipo_decisao) ──────────────────
# Máxima precisão: identifica o acórdão exato pela chave composta
# que também é usada internamente como id_mapa.

registros_completos = {
    ('202202853462', '2023-06-01', 'Acórdão'),
    ('202201555326', '2022-06-20', 'Acórdão'),
}

ckan = UtilCkan(
    registros     = registros_completos, # ← tuplas (registro, data, tipo_decisao)
    download_dir  = DOWNLOAD_DIR,        # ← opcional - padrão é 'downloads_stj'
    timeout       = CKAN_TIMEOUT,        # ← opcional - padrão é 600
    atualizar_cache_e_mapas = ATUALIZAR_CACHE_E_MAPAS,   # ← opcional - padrão é 12h
)

encontrados = ckan.consultar_mapa()
print(f'Registros encontrados (registro + data + tipo): {len(encontrados)}')
for r in encontrados:
    print(f"  id_mapa: {r['id_mapa']}")
    print(f"    orgao: {r['orgao']}  |  classe: {r['sigla_classe']}")

df = ckan.gerar_dataset_espelhos(
    caminho_saida    = PARQUET_SAIDA.replace('.parquet', '_ex_reg_completo.parquet'),
    incluir_integras = True,    # ← busca também o inteiro teor
    incluir_ementas  = True,
    incluir_decisoes = True,
)
UtilCkan.exibir_amostra(df, n=3, titulo='Acórdãos por chave completa (registro+data+tipo)')


  📋  Mapa espelhos carregado: 120239 registros
  📋  Mapa íntegras carregado: 2465080 registros
Registros encontrados (registro + data + tipo): 2
  id_mapa: 202202853462.20230601.ACÓRDÃO
    orgao: S3  |  classe: HC
  id_mapa: 202201555326.20220620.ACÓRDÃO
    orgao: T5  |  classe: HC
Registros cruzados (espelho × íntegra): 2


Espelhos:   0%|          | 0/2 [00:00<?, ?it/s]


Registros extraídos: 2


Extraindo íntegras:   0%|          | 0/2 [00:00<?, ?it/s]

Íntegras extraídas: 2 / 2
───────────────────────────────────────────────────────
  📄  ../data/espelhos_acordaos_por_ano_com_texto_ex_reg_completo.parquet
      0.10 MB | 25 colunas | 2 registros
───────────────────────────────────────────────────────
  Com integra     :      2  (100.0%)
  Sem integra     :      0  (0.0%)
───────────────────────────────────────────────────────
  📅  Por ano de publicação:
      2022 →     1 registros | integra: 1
      2023 →     1 registros | integra: 1
───────────────────────────────────────────────────────
  ⚖️  Top classes:
      HC                                      2
───────────────────────────────────────────────────────
  ✅  Concluído!

Acórdãos por chave completa (registro+data+tipo) (2 registros totais, exibindo 2):
═════════════════════════════════════════════════════════════════
  dataDecisao                 : 20230510
  dataPublicacao              : DJE        DATA:01/06/2023
  data_publicacao_iso         : 20230601
  descricaoClasse     

## 3. Extração de Dados Apenas da Íntegra (UtilCkanIntegra)

A `UtilCkanIntegra` trabalha **exclusivamente** com o dataset de íntegras e seus metadados,
sem depender dos espelhos de acórdão. É útil quando o interesse está apenas no texto integral
e nos campos publicados nos metadados (número de registro, ministro, processo, tipo, etc.).

Aceita os mesmos filtros da `UtilCkan`: `anos`, `datas`, `classes`, `registros`, `documentos`.

In [39]:
# ── UtilCkanIntegra: extração focada apenas nas íntegras ──────────────────────
# Não depende dos espelhos — usa apenas os metadados e textos do dataset de íntegras.
# Ajuste os filtros abaixo conforme necessidade.

ANOS_INTEGRA       = None               # None = todos os anos
DATAS              = {'2023-11-16','2024-06-25'}
REGISTROS_INTEGRA  = None
DOCUMENTOS_INTEGRA = None                   # None = todos; ex: {266239985}
PARQUET_INTEGRAS   = os.path.join(PASTA_DATA, 'integras_por_ano.parquet')

integra = UtilCkanIntegra(
    anos        = ANOS_INTEGRA,
    datas       = DATAS,                   # None = todas as datas
    registros   = REGISTROS_INTEGRA,
    documentos  = DOCUMENTOS_INTEGRA,
    download_dir = DOWNLOAD_DIR,
    timeout      = CKAN_TIMEOUT,
    atualizar_cache_e_mapas = ATUALIZAR_CACHE_E_MAPAS,
)

# Consulta o mapa de íntegras com os filtros aplicados
itens = integra.consultar_mapa()
print(f'Mapa íntegras : {len(integra._mapa_integras)} registros totais')
print(f'Após filtros  : {len(itens)} registros')

# Gera o DataFrame com metadados + texto integral
df = integra.gerar_dataset_integras(
    caminho_saida = PARQUET_INTEGRAS,
    incluir_texto = True,
)

UtilCkanIntegra.exibir_amostra(df, n=2, titulo='Dataset de íntegras')


  📋  Mapa íntegras carregado: 2465080 registros
Mapa íntegras : 2465080 registros totais
Após filtros  : 7831 registros
Registros no mapa de íntegras: 7831
Registros únicos: 7831


Extraindo íntegras:   0%|          | 0/2 [00:00<?, ?it/s]

Íntegras extraídas: 7724 / 7831
Com texto da íntegra: 7724 / 7831
───────────────────────────────────────────────────────
  📄  ../data/integras_por_ano.parquet
      27.35 MB | 11 colunas | 7831 registros
───────────────────────────────────────────────────────
  Com integra     :   7724  (98.6%)
  Sem integra     :    107  (1.4%)
───────────────────────────────────────────────────────
  📅  Por ano de publicação:
      2023 →  3656 registros | integra: 3656
      2024 →  4175 registros | integra: 4068
───────────────────────────────────────────────────────
  ✅  Concluído!

Dataset de íntegras (7831 registros totais, exibindo 2):
═════════════════════════════════════════════════════════════════
  arquivo_integra             : 20231117.zip
  arquivo_metadados           : metadados20231117.json
  data_publicacao             : 20231116
  data_publicacao_iso         : 20231116
  id_mapa                     : 202302589380.20231116.ACÓRDÃO
  ministro                    : GURGEL DE FARIA
  nume